# Distances computed in 4 ways

##


In [1]:
import math
from pyproj import Geod, Transformer

# (lon, lat). I'll choose 2 POI in Almaty
a = (76.8920, 43.2018)   # Mega Alma-Ata
b = (76.9566, 43.2333)   # Dostyk Plaza

def haversine(lon1, lat1, lon2, lat2, R=6_371_008.8):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    h = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(h))

def euclid(epsg):
    tr = Transformer.from_crs("EPSG:4326", epsg, always_xy=True)
    (x1, y1), (x2, y2) = tr.transform(*a), tr.transform(*b)
    return math.hypot(x2 - x1, y2 - y1)

_, _, d_geod = Geod(ellps="WGS84").inv(*a, *b)

results = {
    "(a) haversine":           haversine(*a, *b),
    "(b) Geod: the reference": d_geod,
    "(c) Euclid EPSG:3857":    euclid("EPSG:3857"),
    "(d) Euclid UTM 43N":      euclid("EPSG:32643"),
}
for name, d in results.items():
    print(f"{name:26s} {d:9.1f} m  {100*(d/d_geod - 1):+6.2f}%")

(a) haversine                 6298.6 m   -0.16%
(b) Geod: the reference       6308.6 m   +0.00%
(c) Euclid EPSG:3857          8652.5 m  +37.16%
(d) Euclid UTM 43N            6307.9 m   -0.01%


### now I want to explain why this results appears this way and try to answer the questions

1. (b) Geod: the reference (6308.6 м / 0.00%) 

    It computes the shortest path on the ellipsoid surface(usually WGS84) with the use of some algorithms (like, GeographicLib). The Earth is not a exact sphere, but rather an ellipse, so this method considers that.

    Role: Absolute baseline for comparison, the error rate 0.00%.

2. (d) Euclid UTM 43N (6307.9 м / -0.01%) 

    Cartesian distance with CRS UTM Zone 43 North.

    The error rate is almost 0 (-0.01%): Almaty is placed on 43rd UTM zone. The projection of UTM by itself used to minimize distortion of distance and area inside exact zone. For local tasks (like transport modeling, GIS analysis) this choice is perfect: the primary school-level plane geometry (the Pythagorean theorem) yields a result almost indistinguishable from complex geodesy on an ellipsoid.

3. (a) Haversine (6298.6 м / -0.16%)

    The Haversine computes distance "along the orthodrome" (great-circle arc), but assuming that the Earth is perfect sphere.

    Error rate is small(only 0.16%), because the mean radius of Earth approaximates the real one not bad, but its not enough for precise calculations. The formula is simple and does not require the other packages to run. −0.16% is about 10 m on 6.3 km, so haversine is good enough for ranking nearby places. It just can't see roads, rivers or one-way streets.

4. (c) Euclid EPSG:3857 (8652.5 м / +37.16%) 

    The Standard projection for all maps (Google Maps, OpenStreetMap, Leaflet). It reflects the Earth onto a square sheet of paper.

    Why the error rate is huge (+37.16%): The Mercator projection distorts scale as one moves from the equator toward the poles (in order to preserve angles and object shapes on a rectangular screen). Almaty is located at approximately 43° N latitude, and in this zone, meters in the EPSG:3857 system no longer correspond to actual meters on the ground. The stretch is 1/cos(latitude) = 1/cos(43.2°) ≈ 1.37, which is exactly my +37%. So never rank on EPSG:3857 distances.